In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# add the required imports
import os
import pandas as pd

food_dilvery_path = os.path.join(path, 'Q1_data.csv') # add the the path to the csv file using os.path.join
df_food_delivery = pd.read_csv(food_dilvery_path) # read the file into a dataframe

print(f"Dataset shape: {df_food_delivery.shape}") # print the shape of the dataframe

In [ ]:
# Task 2: Write your code here:
df_food_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_food_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_food_delivery.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black') # create a histogram to plot the distribution of delivery time

  plt.title(f"Target Distribution ({target_column})") # set the title for the plot
  plt.xlabel(target_column) # name the x axis
  plt.ylabel("Frequency") # name the y axis
  plt.grid(False) # don't show th grid

  plt.show() # display the plot

check_target_distribution(df_food_delivery, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df_food_delivery = df_food_delivery.drop(columns="Order_ID", axis=1)

In [ ]:
# Task 2: Write your code here:
cols_with_missing_vals = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food_delivery.copy()
df_clean = df_clean.dropna(subset=['Courier_Experience_yrs', 'Delivery_Time'])
for col in cols_with_missing_vals:
  df_clean[col].fillna(df_food_delivery[col].mode()[0])

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder # import LabelEncoder

#categorical_cols = df_clean.select_dtypes(include=["object"]).columns
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
le = LabelEncoder() # Instantiate LabelEncoder

for col in categorical_cols:
  df_clean[col] = le.fit_transform(df_clean[col])

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

features = df_clean.columns.drop("Delivery_Time") # select only the features for scaling

standard_scaler = StandardScaler() # Instantiate StandardScaler
df_clean[features] = standard_scaler.fit_transform(df_clean[features]) # Apply fit_transform
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# We can't check for target imabalance here because the target is countinuous

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean["Delivery_Time"].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_mae = []
rf_regressor = RandomForestRegressor(n_estimators=200)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/5")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  # Train
  rf_regressor.fit(X_train, y_train)

  # Predict
  y_pred = rf_regressor.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

  rf_mae.append(mae)


In [ ]:
# Task 1: Write your code here:

# Feature importance
feature_importance = pd.DataFrame({
    'feature': list(X),
    'importance': rf_regressor.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black', color='green')
plt.title('Predicted Delivery Time')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: